<div style="background-color:#F3F2EE">
    <br /><br />
        <p style="text-align: center;">
            <font size="6" color='#0A1781'>
                <strong>
                    Databricks Certification Learning Knowledge Graph
                </strong>
              </font>
        </p>
        <p style="text-align: center;">
            <font size="6" color='#C58A1E'>
                <strong>
                    Ingest Learning Evidence from Projects
                </strong>
            </font>
        </p>
        <p style="text-align: center;">
            <font size="5" color='#C58A1E'>
                <strong>
                    Importar evidências reais geradas pelo projeto `databricks-credit-risk-lakehouse`<br />e conectá-las aos tópicos do Knowledge Graph.
                </strong>
            </font>
        </p>
    <br />
</div>

<div style="background-color:#F3F2EE">
    <p style="text-align: right;">
      <font size="4" color='#444444'>
            Roberto SSoares - LfLngLrnng
      </font>
    </p>
    <p style="text-align: right;"><font size="2" color='#444444'>
        <a href="https://www.linkedin.com/in/roberto-dos-santos-soares/">in/roberto-dos-santos-soares</a><br /><a href="https://roberto-ssoares.github.io/meu-portfolio/">Portifólio: roberto-ssoares</a>
    </p>
    <p style="text-align: right;">
        <font size="4" color='#444444'>
            " [+] Faturamento [-] Custo [+] Qualidade de vida "
        </font>
        <br />
        <font size="2" color='#918e8e'>"Mestre Bruno Jardim"
        </font>
    </p>        
    <p style="text-align: right;">        
        <font size="2" color='#918e8e'>           
        </font>
    </p>
</div>

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>📌 Objetivo</strong></font>

<font size="2" color='#66666'>

>- **Ações realizadas**
    - *Leitura da evidência externa.*
    - *Validação dos tópicos no KG.*
    - *Criação de nós `Project`, `Notebook` e `LearningEvidence`.*
    - *Criação de relacionamentos com os tópicos.*
    - *Atualização dos scores de progresso.*
    - *Carga incremental no Neo4j.*

>- **Justificativa técnica**
    - O Knowledge Graph deve evoluir com base em evidências reais de estudo e prática, evitando registrar progresso apenas por intenção ou planejamento.

>- **Resultados esperados**
    - Ao final deste notebook, as evidências reais do Notebook 01 do projeto prático estarão incorporadas ao KG.

---

</font></div>

In [ ]:
#!uv pip install watermark -q -U
#!uv pip install tabulate -q -U
#!uv pip install networkx pyvis matplotlib -q -U
#!uv pip install dotenv
#!uv pip install neo4j -q

In [1]:
from datetime import date
from pathlib import Path
import os
import re

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

In [2]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "RobertoSSoares-LfLngLrnng"

Author: RobertoSSoares-LfLngLrnng



In [3]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [4]:
PROJECT_NAME = "Databricks Certification Learning Knowledge Graph"

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
EVIDENCE_RAW_DIR = RAW_DIR / "evidence"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORTS_DIR = DATA_DIR / "exports"
DOCS_DIR = BASE_DIR / "docs"
CYPHER_DIR = BASE_DIR / "cypher"

for directory in [EVIDENCE_RAW_DIR, PROCESSED_DIR, EXPORTS_DIR, DOCS_DIR, CYPHER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BASE_DIR

WindowsPath('D:/_DS-Projects/Data-Science/databricks-learning-kg')

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 1. Carregar evidência externa</strong></font>

<font size="2" color='#66666'></font></div>

In [24]:
#evidence_path = EVIDENCE_RAW_DIR / "credit_risk_lakehouse_notebook_03_evidence.csv"

#learning_evidence = pd.read_csv(evidence_path).fillna("")

#learning_evidence

In [25]:
evidence_files = sorted(EVIDENCE_RAW_DIR.glob("*_evidence.csv"))

if not evidence_files:
    raise FileNotFoundError(f"Nenhum arquivo de evidência encontrado em: {EVIDENCE_RAW_DIR}")

evidence_frames = []

for path in evidence_files:
    df = pd.read_csv(path).fillna("")
    df["source_evidence_file"] = path.name
    evidence_frames.append(df)

learning_evidence = pd.concat(evidence_frames, ignore_index=True)

# Garante idempotência lógica: se o mesmo evidence_id aparecer mais de uma vez,
# mantemos apenas uma ocorrência.
learning_evidence = (
    learning_evidence
    .drop_duplicates(subset=["evidence_id"], keep="last")
    .reset_index(drop=True)
)

print(f"Arquivos lidos: {len(evidence_files)}")
print(f"Evidências únicas: {len(learning_evidence)}")

learning_evidence


Arquivos lidos: 3
Evidências únicas: 17


,evidence_id,project,notebook,evidence_date,topic,evidence_type,confidence_delta_suggested,mastery_delta_suggested,evidence_description,source_evidence_file
0,EVID_CR_LH_001,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,Lakehouse Architecture,conceptual_study,0.20,0.05,"Comparação entre Data Lake, Data Warehouse e Lakehouse aplicada ao domínio de risco de crédito.",credit_risk_lakehouse_notebook_01_evidence.csv
1,EVID_CR_LH_002,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,Medallion Architecture,architecture_mapping,0.20,0.10,"Mapeamento Raw, Bronze, Silver e Gold para entidades de risco de crédito.",credit_risk_lakehouse_notebook_01_evidence.csv
2,EVID_CR_LH_003,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,Delta Lake,conceptual_study,0.10,0.00,Estudo inicial sobre capacidades do Delta Lake e relevância para confiabilidade dos dados.,credit_risk_lakehouse_notebook_01_evidence.csv
3,EVID_CR_LH_004,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,Managed Table,conceptual_study,0.10,0.00,Diferenciação inicial entre tabela gerenciada e tabela externa.,credit_risk_lakehouse_notebook_01_evidence.csv
4,EVID_CR_LH_005,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,External Table,conceptual_study,0.10,0.00,Diferenciação inicial entre tabela externa e tabela gerenciada.,credit_risk_lakehouse_notebook_01_evidence.csv
5,EVID_CR_LH_006,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,Naming Conventions,project_standard,0.10,0.05,"Definição de convenções iniciais para tabelas, colunas e identificadores.",credit_risk_lakehouse_notebook_01_evidence.csv
6,EVID_CR_LH_007,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,Data Ingestion,raw_layer_preparation,0.15,0.05,Preparação dos arquivos brutos que serão usados como origem para ingestão Raw to Bronze.,credit_risk_lakehouse_notebook_02_evidence.csv
7,EVID_CR_LH_008,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,File Formats,csv_generation,0.10,0.05,"Geração de arquivos CSV para customers, contracts, payments e credit_events.",credit_risk_lakehouse_notebook_02_evidence.csv
8,EVID_CR_LH_009,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,Schema Definition,schema_documentation,0.15,0.05,"Criação de dicionário de dados inicial com tipos esperados, papéis de coluna e nulidade esperada.",credit_risk_lakehouse_notebook_02_evidence.csv
9,EVID_CR_LH_010,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,Data Quality,raw_quality_profile,0.10,0.05,"Geração de perfil inicial de qualidade com nulos, blanks, duplicidades e tipos inferidos.",credit_risk_lakehouse_notebook_02_evidence.csv


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 2. Carregar tópicos do KG</strong></font>

<font size="2" color='#66666'></font></div>

In [26]:
topics = pd.read_csv(PROCESSED_DIR / "topics.csv").fillna("")

topics_lookup = topics[
    ["topic_id", "name", "domain_id", "category", "priority", "confidence_score", "mastery_score"]
].copy()

topics_lookup.head()

,topic_id,name,domain_id,category,priority,confidence_score,mastery_score
0,T001,Lakehouse Architecture,D01,Platform,high,0.25,0.10
1,T002,Databricks Workspace,D01,Platform,medium,0.10,0.00
2,T003,Compute,D01,Platform,medium,0.10,0.00
3,T004,SQL Warehouse,D01,Platform,medium,0.10,0.00
4,T005,Notebooks,D01,Platform,high,0.50,0.25


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 3. Mapeamento robusto por nome do tópico</strong></font>

<font size="2" color='#66666'></font></div>

In [27]:
def normalize_text(value: str) -> str:
    return (
        str(value)
        .strip()
        .lower()
        .replace("-", " ")
        .replace("_", " ")
    )


learning_evidence["topic_key"] = learning_evidence["topic"].apply(normalize_text)
topics_lookup["topic_key"] = topics_lookup["name"].apply(normalize_text)

evidence_mapped = learning_evidence.merge(
    topics_lookup,
    on="topic_key",
    how="left",
    suffixes=("_evidence", "_kg"),
)

missing_topics = evidence_mapped[evidence_mapped["topic_id"].isna()]

if not missing_topics.empty:
    display(missing_topics[["evidence_id", "topic", "topic_key"]])
    raise ValueError("Existem evidências com tópicos não encontrados no KG.")

evidence_mapped[
    [
        "evidence_id",
        "topic",
        "topic_id",
        "name",
        "confidence_delta_suggested",
        "mastery_delta_suggested",
    ]
]


,evidence_id,topic,topic_id,name,confidence_delta_suggested,mastery_delta_suggested
0,EVID_CR_LH_001,Lakehouse Architecture,T001,Lakehouse Architecture,0.20,0.05
1,EVID_CR_LH_002,Medallion Architecture,T020,Medallion Architecture,0.20,0.10
2,EVID_CR_LH_003,Delta Lake,T016,Delta Lake,0.10,0.00
3,EVID_CR_LH_004,Managed Table,T018,Managed Table,0.10,0.00
4,EVID_CR_LH_005,External Table,T019,External Table,0.10,0.00
5,EVID_CR_LH_006,Naming Conventions,T046,Naming Conventions,0.10,0.05
6,EVID_CR_LH_007,Data Ingestion,T007,Data Ingestion,0.15,0.05
7,EVID_CR_LH_008,File Formats,T011,File Formats,0.10,0.05
8,EVID_CR_LH_009,Schema Definition,T013,Schema Definition,0.15,0.05
9,EVID_CR_LH_010,Data Quality,T045,Data Quality,0.10,0.05


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 4. Criar nós novos</strong></font>

<font size="2" color='#66666'></font></div>

In [29]:
project_node = pd.DataFrame(
    [
        {
            "node_id": "PRJ_CR_LAKEHOUSE",
            "node_label": "Project",
            "name": "databricks-credit-risk-lakehouse",
            "category": "Practical Learning Project",
            "description": "Projeto prático de Lakehouse para risco de crédito usado como fonte de evidências reais.",
            "status": "in_progress",
            "priority": "high",
        }
    ]
)

notebook_node = pd.DataFrame(
    [
        {
            "node_id": "NB_CR_LH_01",
            "node_label": "Notebook",
            "name": "01_lakehouse_fundamentals.ipynb",
            "category": "Lakehouse Fundamentals",
            "description": "Notebook prático-conceitual sobre fundamentos de Lakehouse aplicados a risco de crédito.",
            "status": "completed",
            "priority": "high",
        }
    ]
)

learning_evidence_nodes = pd.DataFrame(
    [
        {
            "node_id": row["evidence_id"],
            "node_label": "LearningEvidence",
            "name": row["evidence_id"],
            "category": row["evidence_type"],
            "description": row["evidence_description"],
            "status": "validated",
            "priority": "medium",
            "project": row["project"],
            "notebook": row["notebook"],
            "evidence_date": row["evidence_date"],
            "topic": row["topic"],
            "confidence_delta_suggested": row["confidence_delta_suggested"],
            "mastery_delta_suggested": row["mastery_delta_suggested"],
        }
        for _, row in evidence_mapped.iterrows()
    ]
)

new_nodes = pd.concat(
    [project_node, notebook_node, learning_evidence_nodes],
    ignore_index=True,
)

new_nodes


,node_id,node_label,name,category,description,status,priority,project,notebook,evidence_date,topic,confidence_delta_suggested,mastery_delta_suggested
0,PRJ_CR_LAKEHOUSE,Project,databricks-credit-risk-lakehouse,Practical Learning Project,Projeto prático de Lakehouse para risco de crédito usado como fonte de evidências reais.,in_progress,high,NaN,NaN,NaN,NaN,NaN,NaN
1,NB_CR_LH_01,Notebook,01_lakehouse_fundamentals.ipynb,Lakehouse Fundamentals,Notebook prático-conceitual sobre fundamentos de Lakehouse aplicados a risco de crédito.,completed,high,NaN,NaN,NaN,NaN,NaN,NaN
2,EVID_CR_LH_001,LearningEvidence,EVID_CR_LH_001,conceptual_study,"Comparação entre Data Lake, Data Warehouse e Lakehouse aplicada ao domínio de risco de crédito.",validated,medium,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,Lakehouse Architecture,0.20,0.05
3,EVID_CR_LH_002,LearningEvidence,EVID_CR_LH_002,architecture_mapping,"Mapeamento Raw, Bronze, Silver e Gold para entidades de risco de crédito.",validated,medium,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,Medallion Architecture,0.20,0.10
4,EVID_CR_LH_003,LearningEvidence,EVID_CR_LH_003,conceptual_study,Estudo inicial sobre capacidades do Delta Lake e relevância para confiabilidade dos dados.,validated,medium,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,Delta Lake,0.10,0.00
5,EVID_CR_LH_004,LearningEvidence,EVID_CR_LH_004,conceptual_study,Diferenciação inicial entre tabela gerenciada e tabela externa.,validated,medium,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,Managed Table,0.10,0.00
6,EVID_CR_LH_005,LearningEvidence,EVID_CR_LH_005,conceptual_study,Diferenciação inicial entre tabela externa e tabela gerenciada.,validated,medium,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,External Table,0.10,0.00
7,EVID_CR_LH_006,LearningEvidence,EVID_CR_LH_006,project_standard,"Definição de convenções iniciais para tabelas, colunas e identificadores.",validated,medium,Databricks Credit Risk Lakehouse,01_lakehouse_fundamentals,2026-04-27,Naming Conventions,0.10,0.05
8,EVID_CR_LH_007,LearningEvidence,EVID_CR_LH_007,raw_layer_preparation,Preparação dos arquivos brutos que serão usados como origem para ingestão Raw to Bronze.,validated,medium,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,Data Ingestion,0.15,0.05
9,EVID_CR_LH_008,LearningEvidence,EVID_CR_LH_008,csv_generation,"Geração de arquivos CSV para customers, contracts, payments e credit_events.",validated,medium,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,File Formats,0.10,0.05


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 4. Criar relacionamentos novos</strong></font>

<font size="2" color='#66666'></font></div>

In [46]:
edges = []


def make_edge_id(source_id: str, relationship: str, target_id: str) -> str:
    """
    Cria um edge_id determinístico para evitar duplicidade em reexecuções.
    """
    return f"{source_id}__{relationship}__{target_id}"


def notebook_to_node_id(notebook_name: str) -> str:
    """
    Converte o nome lógico do notebook em um node_id padronizado.
    
    Exemplos:
    01_lakehouse_fundamentals -> NB_CR_LH_01
    02_data_source_and_raw_layer_preparation -> NB_CR_LH_02
    03_raw_to_bronze_ingestion -> NB_CR_LH_03
    """
    notebook_number = str(notebook_name).split("_")[0]
    return f"NB_CR_LH_{notebook_number}"


def add_edge(
    source_id,
    source_label,
    relationship,
    target_id,
    target_label,
    weight=1.0,
    description="",
):
    """
    Adiciona um relacionamento à lista de arestas.
    """
    edges.append(
        {
            "edge_id": make_edge_id(source_id, relationship, target_id),
            "source_id": source_id,
            "source_label": source_label,
            "relationship": relationship,
            "target_id": target_id,
            "target_label": target_label,
            "weight": weight,
            "description": description,
        }
    )


# Projeto possui notebooks utilizados como fonte de evidência
unique_notebooks = sorted(evidence_mapped["notebook"].dropna().unique())

for notebook_name in unique_notebooks:
    notebook_id = notebook_to_node_id(notebook_name)
    
    add_edge(
        "PRJ_CR_LAKEHOUSE",
        "Project",
        "HAS_NOTEBOOK",
        notebook_id,
        "Notebook",
        description=f"Projeto prático possui o notebook {notebook_name}.",
    )


# Relacionamentos por evidência
for _, row in evidence_mapped.iterrows():
    notebook_id = notebook_to_node_id(row["notebook"])
    
    add_edge(
        "PRJ_CR_LAKEHOUSE",
        "Project",
        "GENERATED_EVIDENCE",
        row["evidence_id"],
        "LearningEvidence",
        description="Projeto prático gerou evidência real de aprendizagem.",
    )
    
    add_edge(
        notebook_id,
        "Notebook",
        "GENERATED_EVIDENCE",
        row["evidence_id"],
        "LearningEvidence",
        description=f"Notebook {row['notebook']} gerou evidência real de aprendizagem.",
    )
    
    add_edge(
        row["evidence_id"],
        "LearningEvidence",
        "EVIDENCES",
        row["topic_id"],
        "Topic",
        weight=row["mastery_delta_suggested"],
        description=f"Evidência real conectada ao tópico {row['name']}.",
    )
    
    add_edge(
        notebook_id,
        "Notebook",
        "PRACTICES",
        row["topic_id"],
        "Topic",
        weight=row["mastery_delta_suggested"],
        description=f"Notebook {row['notebook']} pratica ou documenta o tópico {row['name']}.",
    )


new_edges = pd.DataFrame(edges)

new_edges


,edge_id,source_id,source_label,relationship,target_id,target_label,weight,description
0,PRJ_CR_LAKEHOUSE__HAS_NOTEBOOK__NB_CR_LH_01,PRJ_CR_LAKEHOUSE,Project,HAS_NOTEBOOK,NB_CR_LH_01,Notebook,1.00,Projeto prático possui o notebook 01_lakehouse_fundamentals.
1,PRJ_CR_LAKEHOUSE__HAS_NOTEBOOK__NB_CR_LH_02,PRJ_CR_LAKEHOUSE,Project,HAS_NOTEBOOK,NB_CR_LH_02,Notebook,1.00,Projeto prático possui o notebook 02_data_source_and_raw_layer_preparation.
2,PRJ_CR_LAKEHOUSE__HAS_NOTEBOOK__NB_CR_LH_03,PRJ_CR_LAKEHOUSE,Project,HAS_NOTEBOOK,NB_CR_LH_03,Notebook,1.00,Projeto prático possui o notebook 03_raw_to_bronze_ingestion.
3,PRJ_CR_LAKEHOUSE__GENERATED_EVIDENCE__EVID_CR_LH_001,PRJ_CR_LAKEHOUSE,Project,GENERATED_EVIDENCE,EVID_CR_LH_001,LearningEvidence,1.00,Projeto prático gerou evidência real de aprendizagem.
4,NB_CR_LH_01__GENERATED_EVIDENCE__EVID_CR_LH_001,NB_CR_LH_01,Notebook,GENERATED_EVIDENCE,EVID_CR_LH_001,LearningEvidence,1.00,Notebook 01_lakehouse_fundamentals gerou evidência real de aprendizagem.
...,...,...,...,...,...,...,...,...
66,NB_CR_LH_03__PRACTICES__T020,NB_CR_LH_03,Notebook,PRACTICES,T020,Topic,0.10,Notebook 03_raw_to_bronze_ingestion pratica ou documenta o tópico Medallion Architecture.
67,PRJ_CR_LAKEHOUSE__GENERATED_EVIDENCE__EVID_CR_LH_017,PRJ_CR_LAKEHOUSE,Project,GENERATED_EVIDENCE,EVID_CR_LH_017,LearningEvidence,1.00,Projeto prático gerou evidência real de aprendizagem.
68,NB_CR_LH_03__GENERATED_EVIDENCE__EVID_CR_LH_017,NB_CR_LH_03,Notebook,GENERATED_EVIDENCE,EVID_CR_LH_017,LearningEvidence,1.00,Notebook 03_raw_to_bronze_ingestion gerou evidência real de aprendizagem.
69,EVID_CR_LH_017__EVIDENCES__T045,EVID_CR_LH_017,LearningEvidence,EVIDENCES,T045,Topic,0.05,Evidência real conectada ao tópico Data Quality.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 5. Atualizar scores dos tópicos</strong></font>

<font size="2" color='#66666'></font></div>

In [47]:
progress_deltas = (
    evidence_mapped
    .groupby("topic_id", as_index=False)
    .agg(
        confidence_delta=("confidence_delta_suggested", "sum"),
        mastery_delta=("mastery_delta_suggested", "sum"),
        evidence_count=("evidence_id", "count"),
    )
)

topics_updated = topics.merge(
    progress_deltas,
    on="topic_id",
    how="left",
)

topics_updated[["confidence_delta", "mastery_delta", "evidence_count"]] = (
    topics_updated[["confidence_delta", "mastery_delta", "evidence_count"]]
    .fillna(0)
)

topics_updated["previous_confidence_score"] = topics_updated["confidence_score"]
topics_updated["previous_mastery_score"] = topics_updated["mastery_score"]

topics_updated["confidence_score"] = (
    topics_updated["confidence_score"] + topics_updated["confidence_delta"]
).clip(upper=1.0)

topics_updated["mastery_score"] = (
    topics_updated["mastery_score"] + topics_updated["mastery_delta"]
).clip(upper=1.0)

topics_updated["last_review_date"] = date.today().isoformat()

topics_updated["status"] = topics_updated["mastery_score"].apply(
    lambda score: (
        "not_started" if score == 0
        else "mapped" if score < 0.25
        else "practicing" if score < 0.50
        else "consolidating" if score < 0.75
        else "strong"
    )
)

topics_updated[
    [
        "topic_id",
        "name",
        "previous_confidence_score",
        "confidence_score",
        "previous_mastery_score",
        "mastery_score",
        "confidence_delta",
        "mastery_delta",
        "evidence_count",
        "status",
    ]
].sort_values(["evidence_count", "mastery_delta"], ascending=False).head(20)


,topic_id,name,previous_confidence_score,confidence_score,previous_mastery_score,mastery_score,confidence_delta,mastery_delta,evidence_count,status
19,T020,Medallion Architecture,0.50,0.80,0.25,0.45,0.30,0.20,2.0,practicing
6,T007,Data Ingestion,0.25,0.55,0.10,0.25,0.30,0.15,2.0,practicing
10,T011,File Formats,0.50,0.75,0.25,0.40,0.25,0.15,2.0,practicing
12,T013,Schema Definition,0.25,0.50,0.10,0.20,0.25,0.10,2.0,mapped
44,T045,Data Quality,0.40,0.60,0.20,0.30,0.20,0.10,2.0,practicing
45,T046,Naming Conventions,0.60,0.75,0.30,0.40,0.15,0.10,2.0,practicing
36,T037,Pipeline Idempotency,0.25,0.35,0.10,0.20,0.10,0.10,1.0,mapped
0,T001,Lakehouse Architecture,0.25,0.45,0.10,0.15,0.20,0.05,1.0,mapped
15,T016,Delta Lake,0.25,0.35,0.10,0.10,0.10,0.00,1.0,mapped
17,T018,Managed Table,0.10,0.20,0.00,0.00,0.10,0.00,1.0,not_started


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 6. Criar snapshot real W01</strong></font>

<font size="2" color='#66666'></font></div>

In [48]:
snapshot_id = "SNAP_W01_REAL"

snapshot_w01_real = pd.DataFrame(
    [
        {
            "snapshot_id": snapshot_id,
            "snapshot_date": date.today().isoformat(),
            "week_number": 1,
            "overall_progress": round(topics_updated["mastery_score"].mean(), 4),
            "coverage_pct": round((topics_updated["mastery_score"] > 0).mean(), 4),
            "avg_confidence": round(topics_updated["confidence_score"].mean(), 4),
            "avg_mastery": round(topics_updated["mastery_score"].mean(), 4),
            "main_gap": "Ainda faltam práticas reais em ingestão Databricks, Delta Lake, Workflows, Jobs, Tasks e Unity Catalog.",
            "main_achievement": "Primeira evidência real do projeto databricks-credit-risk-lakehouse incorporada ao KG.",
        }
    ]
)

snapshot_node = pd.DataFrame(
    [
        {
            "node_id": snapshot_id,
            "node_label": "Snapshot",
            "name": "Week 01 Real Evidence Snapshot",
            "category": "Learning Progress",
            "description": snapshot_w01_real["main_achievement"].iloc[0],
            "status": "completed",
            "priority": "high",
        }
    ]
)

snapshot_edges = pd.DataFrame(
    [
        {
            "edge_id": f"SNAP_EDGE_{i:04d}",
            "source_id": snapshot_id,
            "source_label": "Snapshot",
            "relationship": "CAPTURES_STATUS_OF",
            "target_id": row["topic_id"],
            "target_label": "Topic",
            "weight": row["mastery_score"],
            "description": f"Snapshot W01 real captura o progresso do tópico {row['name']}.",
        }
        for i, (_, row) in enumerate(topics_updated.iterrows(), start=1)
    ]
)

display(snapshot_w01_real)
display(snapshot_node)
display(snapshot_edges.head())


,snapshot_id,snapshot_date,week_number,overall_progress,coverage_pct,avg_confidence,avg_mastery,main_gap,main_achievement
0,SNAP_W01_REAL,2026-04-29,1,0.1087,0.4783,0.2576,0.1087,"Ainda faltam práticas reais em ingestão Databricks, Delta Lake, Workflows, Jobs, Tasks e Unity Catalog.",Primeira evidência real do projeto databricks-credit-risk-lakehouse incorporada ao KG.


,node_id,node_label,name,category,description,status,priority
0,SNAP_W01_REAL,Snapshot,Week 01 Real Evidence Snapshot,Learning Progress,Primeira evidência real do projeto databricks-credit-risk-lakehouse incorporada ao KG.,completed,high


,edge_id,source_id,source_label,relationship,target_id,target_label,weight,description
0,SNAP_EDGE_0001,SNAP_W01_REAL,Snapshot,CAPTURES_STATUS_OF,T001,Topic,0.15,Snapshot W01 real captura o progresso do tópico Lakehouse Architecture.
1,SNAP_EDGE_0002,SNAP_W01_REAL,Snapshot,CAPTURES_STATUS_OF,T002,Topic,0.00,Snapshot W01 real captura o progresso do tópico Databricks Workspace.
2,SNAP_EDGE_0003,SNAP_W01_REAL,Snapshot,CAPTURES_STATUS_OF,T003,Topic,0.00,Snapshot W01 real captura o progresso do tópico Compute.
3,SNAP_EDGE_0004,SNAP_W01_REAL,Snapshot,CAPTURES_STATUS_OF,T004,Topic,0.00,Snapshot W01 real captura o progresso do tópico SQL Warehouse.
4,SNAP_EDGE_0005,SNAP_W01_REAL,Snapshot,CAPTURES_STATUS_OF,T005,Topic,0.25,Snapshot W01 real captura o progresso do tópico Notebooks.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 7. Exportar artefatos</strong></font>

<font size="2" color='#66666'></font></div>

In [49]:
new_nodes_all = pd.concat([new_nodes, snapshot_node], ignore_index=True)
new_edges_all = pd.concat([new_edges, snapshot_edges], ignore_index=True)

new_nodes_all.to_csv(PROCESSED_DIR / "incremental_nodes_learning_evidence_w01.csv", index=False, encoding="utf-8")
new_edges_all.to_csv(PROCESSED_DIR / "incremental_edges_learning_evidence_w01.csv", index=False, encoding="utf-8")
topics_updated.to_csv(PROCESSED_DIR / "topics_progress_after_credit_risk_lakehouse_nb03.csv", index=False, encoding="utf-8")
snapshot_w01_real.to_csv(PROCESSED_DIR / "snapshot_w01_real.csv", index=False, encoding="utf-8")

evidence_mapped.to_csv(EXPORTS_DIR / "evidence_mapped_credit_risk_lakehouse_nb03.csv", index=False, encoding="utf-8")

print("Artefatos incrementais exportados com sucesso.")


Artefatos incrementais exportados com sucesso.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 8. Conectar ao Neo4j</strong></font>

<font size="2" color='#66666'></font></div>

In [50]:
load_dotenv(BASE_DIR / ".env", override=True)

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("RETURN 'Neo4j connection OK' AS status")
    print(result.single()["status"])
    

Neo4j connection OK


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 9. Funções de carga incremental</strong></font>

<font size="2" color='#66666'></font></div>

In [51]:
def safe_neo4j_name(value: str) -> str:
    value = str(value).strip()
    value = re.sub(r"[^A-Za-z0-9_]", "_", value)
    
    if not value:
        raise ValueError("Nome inválido para Neo4j.")
    
    if value[0].isdigit():
        value = f"_{value}"
    
    return value


def row_to_props(row: pd.Series, exclude: set[str]) -> dict:
    props = {}
    
    for key, value in row.items():
        if key in exclude:
            continue
        
        if pd.isna(value):
            continue
        
        props[key] = value
    
    return props


def load_node(tx, node: dict):
    node_label = safe_neo4j_name(node["node_label"])
    
    cypher = f"""
    MERGE (n:KGNode:{node_label} {{node_id: $node_id}})
    SET n += $props
    """
    
    tx.run(
        cypher,
        node_id=str(node["node_id"]),
        props=node["props"],
    )


def load_relationship(tx, edge: dict):
    relationship = safe_neo4j_name(edge["relationship"])
    
    cypher = f"""
    MATCH (s:KGNode {{node_id: $source_id}})
    MATCH (t:KGNode {{node_id: $target_id}})
    MERGE (s)-[r:{relationship} {{edge_id: $edge_id}}]->(t)
    SET r += $props
    """
    
    tx.run(
        cypher,
        source_id=str(edge["source_id"]),
        target_id=str(edge["target_id"]),
        edge_id=str(edge["edge_id"]),
        props=edge["props"],
    )
    

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 10. Carregar nós incrementais</strong></font>

<font size="2" color='#66666'></font></div>

In [52]:
node_records = []

for _, row in new_nodes_all.iterrows():
    props = row_to_props(row, exclude={"node_id"})
    node_records.append(
        {
            "node_id": str(row["node_id"]),
            "node_label": str(row["node_label"]),
            "props": props,
        }
    )

with driver.session(database=NEO4J_DATABASE) as session:
    for node in node_records:
        session.execute_write(load_node, node)

print(f"Nós incrementais carregados: {len(node_records)}")


Nós incrementais carregados: 20


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 11. Carregar relacionamentos incrementais</strong></font>

<font size="2" color='#66666'></font></div>

In [53]:
edge_records = []

for _, row in new_edges_all.iterrows():
    props = row_to_props(row, exclude={"source_id", "target_id"})
    edge_records.append(
        {
            "edge_id": str(row["edge_id"]),
            "source_id": str(row["source_id"]),
            "target_id": str(row["target_id"]),
            "relationship": str(row["relationship"]),
            "props": props,
        }
    )

with driver.session(database=NEO4J_DATABASE) as session:
    for edge in edge_records:
        session.execute_write(load_relationship, edge)

print(f"Relacionamentos incrementais carregados: {len(edge_records)}")


Relacionamentos incrementais carregados: 117


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 12. Atualizar propriedades dos tópicos no Neo4j</strong></font>

<font size="2" color='#66666'></font></div>

In [54]:
def update_topic_progress(tx, topic: dict):
    cypher = """
    MATCH (t:KGNode:Topic {node_id: $topic_id})
    SET
        t.confidence_score = $confidence_score,
        t.mastery_score = $mastery_score,
        t.status = $status,
        t.last_review_date = $last_review_date
    """
    
    tx.run(
        cypher,
        topic_id=str(topic["topic_id"]),
        confidence_score=float(topic["confidence_score"]),
        mastery_score=float(topic["mastery_score"]),
        status=str(topic["status"]),
        last_review_date=str(topic["last_review_date"]),
    )


topic_progress_records = topics_updated[
    ["topic_id", "confidence_score", "mastery_score", "status", "last_review_date"]
].to_dict("records")

with driver.session(database=NEO4J_DATABASE) as session:
    for topic in topic_progress_records:
        session.execute_write(update_topic_progress, topic)

print(f"Tópicos atualizados no Neo4j: {len(topic_progress_records)}")


Tópicos atualizados no Neo4j: 46


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 13. Consultas de validação</strong></font>

<font size="2" color='#66666'></font></div>

In [55]:
queries = {
    "learning_evidence_nodes": """
        MATCH (e:LearningEvidence)
        RETURN count(e) AS total
    """,
    "evidence_relationships": """
        MATCH (:LearningEvidence)-[r:EVIDENCES]->(:Topic)
        RETURN count(r) AS total
    """,
    "project_evidence_paths": """
        MATCH path = (:Project {node_id: 'PRJ_CR_LAKEHOUSE'})-[:GENERATED_EVIDENCE]->(:LearningEvidence)-[:EVIDENCES]->(:Topic)
        RETURN count(path) AS total
    """,
}

validation = []

with driver.session(database=NEO4J_DATABASE) as session:
    for name, query in queries.items():
        result = session.run(query)
        validation.append(
            {
                "check": name,
                "total": result.single()["total"],
            }
        )

validation_df = pd.DataFrame(validation)

validation_df


,check,total
0,learning_evidence_nodes,19
1,evidence_relationships,45
2,project_evidence_paths,123


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 14. Consulta para o Neo4j Browser</strong></font>

<font size="2" color='#66666'></font></div>

In [56]:
browser_query = """
MATCH path = (:Project {node_id: 'PRJ_CR_LAKEHOUSE'})
             -[:GENERATED_EVIDENCE]->(:LearningEvidence)
             -[:EVIDENCES]->(:Topic)
RETURN path
LIMIT 100;
"""

print(browser_query)



MATCH path = (:Project {node_id: 'PRJ_CR_LAKEHOUSE'})
             -[:GENERATED_EVIDENCE]->(:LearningEvidence)
             -[:EVIDENCES]->(:Topic)
RETURN path
LIMIT 100;



<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 15. Relatório executivo</strong></font>

<font size="2" color='#66666'></font></div>

In [57]:
report = f"""# Learning Evidence Ingestion W01 — Databricks Learning KG

## Projeto

{PROJECT_NAME}

## Fonte da evidência

Projeto: `databricks-credit-risk-lakehouse`  
Notebook: `01_lakehouse_fundamentals.ipynb`  
Data: {date.today().isoformat()}

## Evidências incorporadas

{evidence_mapped[["evidence_id", "topic", "topic_id", "evidence_type", "confidence_delta_suggested", "mastery_delta_suggested"]].to_markdown(index=False)}

## Validação Neo4j

{validation_df.to_markdown(index=False)}

## Snapshot W01 real

{snapshot_w01_real.to_markdown(index=False)}

## Leitura executiva

O Knowledge Graph passou a incorporar evidências reais vindas do projeto prático `databricks-credit-risk-lakehouse`.

A partir deste ponto, a evolução dos tópicos da certificação Databricks será baseada em artefatos concretos: notebooks, estudos aplicados, práticas, cargas, consultas e entregáveis de portfólio.

Esta etapa estabelece a ponte entre estudo prático e monitoramento semântico da jornada de aprendizagem.
"""

report_path = DOCS_DIR / "learning_evidence_ingestion_w01.md"
report_path.write_text(report, encoding="utf-8")

report_path


WindowsPath('D:/_DS-Projects/Data-Science/databricks-learning-kg/docs/learning_evidence_ingestion_w01.md')

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 16. Encerrar conexão</strong></font>

<font size="2" color='#66666'></font></div>

In [58]:
driver.close()

print("Conexão com Neo4j encerrada.")

Conexão com Neo4j encerrada.


In [59]:
%reload_ext watermark 
%watermark -a "Roberto-SSoares-LfLngLrnng" -d -t -u --iversions -v -m -h

Author: Roberto-SSoares-LfLngLrnng

Last updated: 2026-04-29 01:08:01

Python implementation: CPython
Python version       : 3.12.12
IPython version      : 9.13.0

Compiler    : MSC v.1944 64 bit (AMD64)
OS          : Windows
Release     : 11
Machine     : AMD64
Processor   : Intel64 Family 6 Model 158 Stepping 9, GenuineIntel
CPU cores   : 4
Architecture: 64bit

Hostname: PC-ROBERTO

dotenv: 0.9.9
neo4j : 6.1.0
pandas: 3.0.2
re    : 2.2.1



<div style="background-color:#f3f2ee">
    
<font size="6" color='#CC403E'><strong>Fim</strong></font>

<font size="2" color='#66666'></font></div>

In [ ]:
#!uv pip install nbconvert -U -q
!jupyter nbconvert --to html --template-file my-template-html-v10.tpl 05_ingest_learning_evidence_from_projects.ipynb